# Notebook 04 — Pattern Ranking, Hotspot Integration, Corridors, Rare Mobility, Planning & KPIs

**Project:** Urban Taxi Mobility Pattern Discovery (Microsoft T-Drive)

**Objective.** Take the frequent PrefixSpan patterns mined in Notebook 03,
rank them by a composite evidence score, integrate them with DBSCAN
hotspots to identify recurring hotspot-to-hotspot mobility corridors,
characterize rare/unusual mobility transitions, and compile planning
insights and KPIs.

**Fixes carried into this version (do not regress):**
- Imports the project's own `src/sequential.py` — `sys.path.insert(0, ...)`,
  not `append(...)` — to avoid shadowing by an unrelated installed package
  of the same name.
- `pattern_ranking.build_enriched_sequences()` groups by
  **taxi_id + calendar date** (keys `"taxi_id::date"`), matching
  `sequential.build_taxi_sequences(split_by_date=True)` exactly, so
  `len(enriched_sequences) == len(sequences)`.
- The corridor pipeline (`identify_corridors` -> `corridor_summary`) uses an
  explicit, consistent schema and **never merges on `support_count`** (not
  a unique key — a prior version merged on it and triggered an
  ~110GB memory blow-up from an accidental many-to-many join).
- Zero identified corridors is treated as a valid result: `corridor_summary()`
  returns an empty DataFrame with the expected schema, and every
  downstream section (planning insights, KPIs) is written to run correctly
  when `corridor_table` is empty.
- Rare/unusual mobility transitions use a transparent, occurrence-based
  primary definition (a transition seen in at most
  `config.RARE_TRANSITION_MAX_TAXI_DAYS` distinct taxi-days), with the
  percentile-based rule kept only as a reported sensitivity comparison.


## 1. Imports and source-path validation

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))  # project src FIRST — not sys.path.append(...)

import config
import spatial
import clustering
import sequential
import pattern_ranking
import integration
import anomaly
import planning

print("sequential loaded from:", sequential.__file__)
print("pattern_ranking loaded from:", pattern_ranking.__file__)
assert str(SRC_DIR) in sequential.__file__, (
    "sequential was NOT loaded from the project's src/ directory — "
    "check sys.path ordering and for a same-named installed package."
)


sequential loaded from: c:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\src\sequential.py
pattern_ranking loaded from: c:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\src\pattern_ranking.py


## 2. Load processed data and Notebook 03 outputs

Loads Notebook 02's two saved outputs — `labeled_sample.csv` (cleaned,
labeled GPS points) and `clustered_sample.csv` (the same points with
DBSCAN `cluster_id` already assigned) — plus Notebook 03's saved
frequent-pattern table, so this notebook does not silently drift onto a
different sample or recompute DBSCAN with different parameters than
Notebook 02 already used.


In [2]:
LABELED_DATA_FILENAME = "labeled_sample.csv"
CLUSTERED_DATA_FILENAME = "clustered_sample.csv"

labeled_path = config.PROCESSED_DATA_DIR / LABELED_DATA_FILENAME
clustered_path = config.PROCESSED_DATA_DIR / CLUSTERED_DATA_FILENAME

for path in (labeled_path, clustered_path):
    if not path.exists():
        candidates = sorted(config.PROCESSED_DATA_DIR.glob("*"))
        raise FileNotFoundError(
            f"Expected processed data at {path}, which does not exist.\n"
            f"Files actually present in {config.PROCESSED_DATA_DIR}:\n"
            + "\n".join(f"  - {p.name}" for p in candidates)
        )

traj_df = pd.read_csv(labeled_path, parse_dates=[config.COL_TIMESTAMP])
clustered_sample_df = pd.read_csv(clustered_path, parse_dates=[config.COL_TIMESTAMP])

if clustering.COL_CLUSTER_ID not in clustered_sample_df.columns:
    raise ValueError(
        f"{clustered_path} does not contain a '{clustering.COL_CLUSTER_ID}' column — "
        f"expected Notebook 02's DBSCAN cluster assignments."
    )

nb03_patterns_path = config.TABLES_DIR / "nb03_frequent_patterns.csv"
if not nb03_patterns_path.exists():
    raise FileNotFoundError(
        f"Expected Notebook 03 output at {nb03_patterns_path} — run Notebook 03 first."
    )
patterns_from_disk = pd.read_csv(nb03_patterns_path)
patterns_from_disk["pattern"] = patterns_from_disk["pattern"].str.split(" -> ").apply(tuple)

print(
    f"Loaded {len(traj_df):,} labeled GPS rows, {len(clustered_sample_df):,} clustered GPS rows, "
    f"and {len(patterns_from_disk):,} Notebook 03 patterns."
)


Loaded 116,913 labeled GPS rows, 116,913 clustered GPS rows, and 22 Notebook 03 patterns.


## 3. Validate taxi-day sequence construction

In [3]:
if spatial.COL_CELL_ID not in traj_df.columns:
    traj_df = spatial.assign_spatial_cells(traj_df, cell_size_meters=config.GRID_CELL_SIZE_METERS)

labeled_df = sequential.add_temporal_labels(traj_df)
sequences = sequential.build_taxi_sequences(labeled_df, split_by_date=True)
n_sequences = len(sequences)
print(f"Taxi-day sequences: {n_sequences:,}")

# Re-mine at the primary threshold rather than trusting the CSV round-trip
# for pattern *tuples* (string round-tripping a tuple of cell ids is done
# above for display, but the ranking step below needs the DataFrame
# produced directly by mine_patterns(), with its native dtypes).
patterns_df = sequential.mine_patterns(
    sequences, min_support=config.DEFAULT_MIN_SUPPORT, max_pattern_length=config.MAX_PATTERN_LENGTH
)
assert len(patterns_df) == len(patterns_from_disk), (
    "Re-mined pattern count does not match Notebook 03's saved output — "
    "check that both notebooks are using the same processed data and config."
)
print("Re-mined pattern count matches Notebook 03's saved output.")


Taxi-day sequences: 630
Re-mined pattern count matches Notebook 03's saved output.


## 4. Build enriched taxi-day sequences

Mirrors `sequential.build_taxi_sequences(split_by_date=True)` exactly —
same grouping, same keys — so every plain taxi-day sequence has a matching
enriched (timestamp/segment/day_type-carrying) sequence.


In [4]:
enriched_sequences = pattern_ranking.build_enriched_sequences(labeled_df, split_by_date=True)

assert len(enriched_sequences) == len(sequences), (
    f"enriched ({len(enriched_sequences)}) vs plain ({len(sequences)}) taxi-day sequence count mismatch "
    f"— check build_enriched_sequences() grouping."
)
sample_key = next(iter(enriched_sequences))
print(f"Enriched sequences: {len(enriched_sequences):,} (matches plain taxi-day sequences)")
print(f"Example key: {sample_key!r}")


Enriched sequences: 630 (matches plain taxi-day sequences)
Example key: '107::2008-02-02'


## 5. Frequent patterns (bounded PrefixSpan, from Section 3)

Notebook 04 reuses the primary-analysis pattern set already mined above
rather than re-running the more expensive sensitivity/subgroup mining
already done in Notebook 03.


In [5]:
print(f"Frequent patterns for ranking: {len(patterns_df):,}")
patterns_df.head()


Frequent patterns for ranking: 22


,pattern,support_count,support_percentage,pattern_length
0,"(CELL_02242, CELL_02242)",43,6.825397,2
1,"(CELL_03758, CELL_03757)",42,6.666667,2
2,"(CELL_03719, CELL_03758)",40,6.349206,2
3,"(CELL_01519, CELL_01520)",40,6.349206,2
4,"(CELL_03719, CELL_03757)",38,6.031746,2


## 6. Rank patterns

In [6]:
grid_summary = spatial.cell_grid_summary(labeled_df)
ranked_df = pattern_ranking.rank_patterns(patterns_df, enriched_sequences, grid_summary)
ranked_df[[
    "pattern", "composite_score", "support_percentage",
    "n_taxi_days_matched", "n_taxis_matched", "dominant_time_segment",
]].head(10)


,pattern,composite_score,support_percentage,n_taxi_days_matched,n_taxis_matched,dominant_time_segment
0,"(CELL_03676, CELL_03719)",0.607622,5.555556,35,21,night
1,"(CELL_02242, CELL_02242)",0.538369,6.825397,43,29,afternoon
2,"(CELL_03758, CELL_03757)",0.535958,6.666667,42,27,afternoon
3,"(CELL_03719, CELL_03758)",0.535823,6.349206,40,26,afternoon
4,"(CELL_03757, CELL_03758)",0.519176,5.079365,32,21,afternoon
5,"(CELL_02242, CELL_02611)",0.517966,5.238095,33,19,afternoon
6,"(CELL_01520, CELL_01521)",0.515508,5.238095,33,29,morning
7,"(CELL_03584, CELL_03758)",0.513711,5.555556,35,26,afternoon
8,"(CELL_03719, CELL_03757)",0.512041,6.031746,38,24,afternoon
9,"(CELL_03584, CELL_03719)",0.506851,5.079365,32,23,afternoon


## 7. Validate: n_taxi_days_matched == support_count

In [7]:
mismatches = ranked_df[ranked_df["n_taxi_days_matched"] != ranked_df["support_count"]]
assert mismatches.empty, f"n_taxi_days_matched/support_count mismatch found:\n{mismatches}"
print("Consistency check passed: n_taxi_days_matched == support_count for every ranked pattern.")


Consistency check passed: n_taxi_days_matched == support_count for every ranked pattern.


## 8. Integrate hotspots (DBSCAN)

Uses Notebook 02's already-computed DBSCAN cluster assignments
(`clustered_sample.csv`, loaded in Section 2) rather than re-running
DBSCAN here — the clustering methodology (haversine distance, configured
eps/min_samples) is unchanged, this notebook simply consumes the existing
output instead of recomputing it a second time with a possibly different
call. Spatial cells are assigned to the clustered data if not already
present, using the same 500m grid as Notebook 03.


In [8]:
if spatial.COL_CELL_ID not in clustered_sample_df.columns:
    clustered_sample_df = spatial.assign_spatial_cells(
        clustered_sample_df, cell_size_meters=config.GRID_CELL_SIZE_METERS
    )
clustered_df = clustered_sample_df

cluster_summary_df = clustering.cluster_summary(clustered_df)
print(f"Clusters found (including the noise row, cluster_id == -1): {len(cluster_summary_df)}")

cell_to_cluster_df = integration.map_cells_to_clusters(grid_summary, clustered_df)
cell_to_cluster = cell_to_cluster_df.set_index(spatial.COL_CELL_ID)["dominant_cluster_id"].to_dict()
cluster_summary_df.head()


Clusters found (including the noise row, cluster_id == -1): 238


,cluster_id,n_points,center_lat,center_lon,is_noise
0,0,69538,39.938308,116.401149,False
1,-1,7586,40.016313,116.500179,True
2,88,1409,39.921224,116.164884,False
3,147,1358,40.063125,116.139841,False
4,21,1290,40.021781,116.263353,False


## 9. Identify mobility corridors

A corridor is a frequent PrefixSpan pattern whose cells map onto **two or
more distinct, non-noise DBSCAN clusters** — i.e. it actually connects two
hotspots, not just recurs within one. Thresholds are never lowered and
corridors are never manufactured to force a nonzero result.


In [9]:
corridors_df = integration.identify_corridors(
    ranked_df, cell_to_cluster, min_clusters_in_corridor=2
)
corridors_only = corridors_df[corridors_df["is_corridor"]]

print(f"Corridors identified: {len(corridors_only)} / {len(corridors_df)} frequent patterns qualify.")


Corridors identified: 0 / 22 frequent patterns qualify.


## 10. Build the corridor table (safe for the zero-corridor case)

`corridor_summary()` returns a DataFrame with a fixed, explicit schema —
including `n_taxi_days_matched`, `n_taxis_matched`, and
`dominant_time_segment` carried directly from the ranked-pattern columns
(never re-joined via a `support_count` merge) — and returns that schema
**empty**, rather than raising, when there are zero corridors.


In [10]:
corridor_table = integration.corridor_summary(corridors_only, cluster_summary_df)

if len(corridor_table) == 0:
    print(
        "No frequent hotspot-to-hotspot mobility corridors were identified "
        "under the configured 5% support threshold and current DBSCAN "
        "hotspot definition."
    )
else:
    print(f"Corridor table built: {len(corridor_table)} corridor(s).")

corridor_table.head()


No frequent hotspot-to-hotspot mobility corridors were identified under the configured 5% support threshold and current DBSCAN hotspot definition.


,cell_pattern,cluster_path,n_distinct_clusters,support_count,support_percentage,n_taxi_days_matched,n_taxis_matched,dominant_time_segment


## 11. Temporal activity analysis

In [11]:
segment_summary = (
    labeled_df.groupby("time_segment").size().reset_index(name="n_points").sort_values("n_points", ascending=False)
)
segment_summary


,time_segment,n_points
3,night,36703
0,afternoon,32193
2,morning,24287
1,evening,23730


## 12. Weekday/weekend activity

In [12]:
day_type_summary = labeled_df.groupby("day_type").size().reset_index(name="n_points")
day_type_summary


,day_type,n_points
0,weekday,84985
1,weekend,31928


## 13. Rare/unusual mobility analysis

**Primary definition** (occurrence-based, transparent): a cell-to-cell
transition observed in at most `config.RARE_TRANSITION_MAX_TAXI_DAYS`
distinct taxi-day sequences is a "rare mobility transition".

A supplementary percentile-based view is also reported, with its
threshold and resulting flagged share made explicit, since on real
taxi-mobility data many distinct transitions occur only once and a
percentile cut can therefore flag a much larger share of transitions than
the percentile number itself suggests.

This analysis is exploratory/descriptive. It does **not** identify fraud,
accidents, incidents, or dangerous routes — the dataset does not support
those claims.


In [13]:
transition_table = anomaly.build_transition_frequency_table(enriched_sequences)

rare_by_occurrence = anomaly.identify_unusual_transitions_by_occurrence(
    transition_table, max_taxi_days=config.RARE_TRANSITION_MAX_TAXI_DAYS
)
rare_by_percentile = anomaly.identify_unusual_transitions_by_percentile(
    transition_table, percentile=config.UNUSUAL_TRANSITION_PERCENTILE
)

n_total = len(transition_table)
print(
    f"Distinct observed transitions: {n_total:,}\n"
    f"Primary rule (n_taxi_days <= {config.RARE_TRANSITION_MAX_TAXI_DAYS}): "
    f"{len(rare_by_occurrence):,} flagged "
    f"({100 * len(rare_by_occurrence) / n_total:.1f}% of distinct transitions)\n"
    f"Supplementary rule (percentile <= {config.UNUSUAL_TRANSITION_PERCENTILE}): "
    f"{len(rare_by_percentile):,} flagged "
    f"({100 * len(rare_by_percentile) / n_total:.1f}% of distinct transitions, "
    f"count threshold = {rare_by_percentile['threshold_count'].iloc[0] if len(rare_by_percentile) else 'N/A'})"
)

rare_occurrences = anomaly.find_unusual_occurrences(enriched_sequences, rare_by_occurrence)
print(f"Individual occurrences of a primary-rule rare transition: {len(rare_occurrences):,}")
rare_occurrences.head()


Distinct observed transitions: 35,365
Primary rule (n_taxi_days <= 2): 31,765 flagged (89.8% of distinct transitions)
Supplementary rule (percentile <= 5.0): 26,638 flagged (75.3% of distinct transitions, count threshold = 1.0)
Individual occurrences of a primary-rule rare transition: 37,141


,taxi_id,from_cell,to_cell,timestamp,time_segment,day_type
0,6874,CELL_03728,CELL_03657,2008-02-02 13:32:44,afternoon,weekend
1,4340,CELL_03158,CELL_03195,2008-02-02 13:34:37,afternoon,weekend
2,7518,CELL_05019,CELL_05007,2008-02-02 13:34:41,afternoon,weekend
3,9126,CELL_02568,CELL_02511,2008-02-02 13:34:45,afternoon,weekend
4,3812,CELL_01943,CELL_01942,2008-02-02 13:34:48,afternoon,weekend


If the percentile-based share above looks high relative to the stated
percentile, that is expected when many distinct transitions are tied at
the same low occurrence count (common in a taxi-day sample) — it is a
property of the count distribution, not a sign that a large share of
mobility is anomalous in any safety-relevant sense.


## 14. Planning insights

Every insight below cites a concrete number from the mined results and is
phrased as something that "may warrant further investigation" — never as
a directive, an optimization claim, or a fleet-size assertion.


In [14]:
insights_df = planning.compile_planning_insights(
    cluster_summary_df, corridor_table, segment_summary, day_type_summary
)
for _, row in insights_df.iterrows():
    print(f"[{row['category']}] {row['insight']}\n")


[recurring taxi hotspot] Cluster 0 (centered near 39.9383, 116.4011) recorded 69,538 GPS observations in the sample — among the busiest spatial clusters found. This area may warrant further investigation for taxi pickup infrastructure because of its sustained observed activity volume.

[recurring taxi hotspot] Cluster 88 (centered near 39.9212, 116.1649) recorded 1,409 GPS observations in the sample — among the busiest spatial clusters found. This area may warrant further investigation for taxi pickup infrastructure because of its sustained observed activity volume.

[recurring taxi hotspot] Cluster 147 (centered near 40.0631, 116.1398) recorded 1,358 GPS observations in the sample — among the busiest spatial clusters found. This area may warrant further investigation for taxi pickup infrastructure because of its sustained observed activity volume.

[time-dependent pickup demand area] The 'night' segment accounted for 36,703 of 116,913 GPS observations in the sample (31.4%). Demand-res

## 15. KPIs

In [15]:
kpis_df = planning.compute_kpis(labeled_df, cluster_summary_df, patterns_df, corridor_table, segment_summary)
kpis_df.T


,0
total_taxis_analyzed,99
total_gps_points,116913
total_spatial_cells,5413
total_spatial_clusters,237
total_frequent_sequences,22
top_sequence,CELL_02242 -> CELL_02242
top_sequence_support_pct,6.83
top_hotspot,Cluster 0
peak_mobility_hour,14
most_active_period,night


## 16. Sensitivity / evaluation

Reuses Notebook 03's saved support-sensitivity table rather than
re-running an expensive mining pass; every threshold there was already
bounded by `MAX_PATTERN_LENGTH` and used the `ceil()`-based count.


In [16]:
sensitivity_path = config.TABLES_DIR / "nb03_support_sensitivity.csv"
if sensitivity_path.exists():
    sensitivity_df = pd.read_csv(sensitivity_path)
    display_cols = [c for c in ["min_support", "min_support_count", "n_patterns", "is_primary_analysis"] if c in sensitivity_df.columns]
    print(sensitivity_df[display_cols])
else:
    print(f"No sensitivity table found at {sensitivity_path} — run Notebook 03 first.")


   min_support  min_support_count  n_patterns  is_primary_analysis
0         0.01                  7       39785                False
1         0.02                 13        3426                False
2         0.05                 32          22                 True


## 17. Save final Phase 4 outputs

In [17]:
config.ensure_output_dirs()

ranked_out = config.TABLES_DIR / "nb04_ranked_patterns.csv"
corridor_out = config.TABLES_DIR / "nb04_corridor_table.csv"
rare_transitions_out = config.TABLES_DIR / "nb04_rare_transitions.csv"
insights_out = config.TABLES_DIR / "nb04_planning_insights.csv"
kpis_out = config.RESULTS_DIR / "nb04_kpis.csv"

ranked_df.assign(pattern=ranked_df["pattern"].apply(" -> ".join)).to_csv(ranked_out, index=False)
corridor_table.to_csv(corridor_out, index=False)
rare_by_occurrence.to_csv(rare_transitions_out, index=False)
insights_df.to_csv(insights_out, index=False)
kpis_df.to_csv(kpis_out, index=False)

print("Saved:")
for p in [ranked_out, corridor_out, rare_transitions_out, insights_out, kpis_out]:
    print(f"  - {p}")


Saved:
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb04_ranked_patterns.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb04_corridor_table.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb04_rare_transitions.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\tables\nb04_planning_insights.csv
  - C:\Users\user\Downloads\urban-taxi-mobility\urban-taxi-mobility\outputs\results\nb04_kpis.csv


## 18. Final interpretation and limitations

- **Corridors.** A count of zero corridors is a valid, reported result
  under the current 5% support threshold and DBSCAN hotspot definition —
  it means no frequent sequence happened to connect two distinct hotspot
  clusters, not that the pipeline failed. Lowering support or DBSCAN
  thresholds specifically to force a nonzero corridor count would bias the
  result and was deliberately not done here.
- **Rare mobility transitions.** Reported as "rare mobility transitions" /
  "unusual mobility behavior" only — never as fraud, accidents, incidents,
  or dangerous routes, and never used to claim traffic congestion. The
  data supports a descriptive frequency claim, not a causal or
  safety-critical one.
- **Planning insights.** Phrased as "may warrant further investigation" /
  "data-driven suggested allocation" where relevant — never as "optimal
  taxi allocation" (no optimization model is implemented here) and never
  assuming a specific fleet size.
- **Support denominators.** Every support figure in this notebook is
  against the ORIGINAL taxi-day sequence count for its group (overall,
  weekday, weekend, or time-segment) — never GPS points, distinct taxis,
  or a post-pruning sequence count.
- **Sample size.** Results reflect the project's development sample;
  absolute counts (corridors, rare transitions, KPI totals) should be
  re-validated on a full T-Drive run before being treated as final.
